In [0]:
acc_name = "medallionazure"
acc_key = dbutils.secrets.get(scope="azure-storage", key="storagekeybdg")
spark.conf.set(f"fs.azure.account.key.{acc_name}.blob.core.windows.net", acc_key)

eh_conn_str = dbutils.secrets.get(scope="azure-storage", key="eventhubs-connection-string")
eh_name = "taxi-rides"

eh_conf = {
    "eventhubs.connectionString": sc._jvm.org.apache.spark.eventhubs.EventHubsUtils.encrypt(
        f"{eh_conn_str};EntityPath={eh_name}"
    ),
    "eventhubs.startingPosition": '{"offset": "-1", "seqNo": -1, "enqueuedTime": null, "isInclusive": true}'
}

streaming_bronze_path = f"wasbs://data@{acc_name}.blob.core.windows.net/streaming/bronze"
checkpoint_path = f"wasbs://data@{acc_name}.blob.core.windows.net/streaming/_checkpoints/bronze"

print("✅ Konfiguracja załadowana")

In [0]:
from pyspark.sql.functions import from_json, col, current_timestamp
from pyspark.sql.types import *

# Schemat danych taxi (musi pasować do tego co wysłał producent)
taxi_schema = StructType([
    StructField("VendorID", LongType()),
    StructField("tpep_pickup_datetime", StringType()),
    StructField("tpep_dropoff_datetime", StringType()),
    StructField("passenger_count", LongType()),
    StructField("trip_distance", DoubleType()),
    StructField("PULocationID", LongType()),
    StructField("DOLocationID", LongType()),
    StructField("payment_type", LongType()),
    StructField("fare_amount", DoubleType()),
    StructField("tip_amount", DoubleType()),
    StructField("total_amount", DoubleType())
])

# Czytaj strumień z Event Hubs
raw_stream = spark.readStream \
    .format("eventhubs") \
    .options(**eh_conf) \
    .load()

# Parsuj JSON z kolumny "body"
parsed_stream = raw_stream \
    .select(
        col("enqueuedTime").alias("event_time"),
        from_json(col("body").cast("string"), taxi_schema).alias("data")
    ) \
    .select("event_time", "data.*") \
    .withColumn("ingestion_timestamp", current_timestamp())

# Zapisuj strumieniowo do Bronze jako Delta
query = parsed_stream.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", checkpoint_path) \
    .start(streaming_bronze_path)

print("🔴 Streaming AKTYWNY — czyta z Event Hubs i zapisuje do Bronze")
print("   Aby zatrzymać: query.stop()")

In [0]:
# Sprawdź ile rekordów już wylądowało
streaming_bronze_df = spark.read.format("delta").load(streaming_bronze_path)
print(f"📊 Rekordy w Streaming Bronze: {streaming_bronze_df.count()}")
display(streaming_bronze_df.limit(10))

In [0]:
query.stop()
print("⏹️ Streaming zatrzymany")